# Lab 4: Geocoding I

<p align="center"><img src="https://raw.githubusercontent.com/geopy/geopy/master/docs/_static/logo-wide.png" width="250"/></p>

In this lab session, we'll:
- Learn how to use the [GeoPy](https://geopy.readthedocs.io/en/stable/) package to perform geocoding, i.e., relate place names to geographic coordinates
- Explore place name ambiguity and use contextual clues to resolve it
- Learn how to compute evaluation metrics: Mean Error Distance (MED) and Median Error Distance (MdnED)
```text

Geocoding      Vienna   → (48.21, 16.37)
               Salzburg → (47.80, 13.04)
               (coordinates found by GeoPy)
                                        │
                                        ▼
Evaluation     Error distance = great-circle distance to the ground truth
               (compare the coordinates with the ground truth)
```

Below I will provide code along with explanations.

## Part 1: Geocoding with GeoPy

In the lecture, we learned that **geocoding** relates place names to positions on the Earth. In this part, we'll geocode one example place name: **Vienna**.

A **geocoder** is a web service that takes a place name and returns its geographic coordinates. Each geocoder relies on a gazetteer behind the scenes.

[GeoPy](https://geopy.readthedocs.io/en/stable/) gives us one Python class for each geocoder in `geopy.geocoders`. We'll try three of them:
- `Nominatim`: based on [OpenStreetMap](https://www.openstreetmap.org/)
- `ArcGIS`: based on [Esri's ArcGIS](https://www.esri.com/en-us/arcgis/products/arcgis-world-geocoder)
- `GeoNames`: based on [GeoNames](https://www.geonames.org/)

### 1.1 Nominatim

Nominatim is free, but its [usage policy](https://operations.osmfoundation.org/policies/nominatim/) asks for two things:
- `user_agent`: a name for your application, so that OpenStreetMap knows what application you are sending requests for.
- At most **1 request per second**. So please don't run the cells too quickly in a row.

Replace `"YourAppName"` below with your application name.

In [ ]:
from geopy.geocoders import Nominatim

# Create a Nominatim geocoder with your application name
nominatim = Nominatim(user_agent="YourAppName")

Each geocoder has a `geocode()` method. We give it a place name, and it gives back a **location** with:
- `location.address`: the full name of the place found
- `location.latitude`: its latitude
- `location.longitude`: its longitude

In [ ]:
# Geocode one place name
location = nominatim.geocode("Vienna")

print(location.address)
print(location.latitude, location.longitude)

### 1.2 ArcGIS

ArcGIS doesn't need an account for basic geocoding. Note that the `geocode()` method works in the same way as with Nominatim.

In [ ]:
from geopy.geocoders import ArcGIS

# Create an ArcGIS geocoder
arcgis = ArcGIS()

# Geocode "Vienna" with ArcGIS
location = arcgis.geocode("Vienna")

print(location.address)
print(location.latitude, location.longitude)

### 1.3 GeoNames

GeoNames needs a **free account**. Please do the following once, before running the code:
1. Register at [geonames.org/login](https://www.geonames.org/login).
2. Confirm your account through the email you receive.

Without the above steps, the code below won't work. Then replace `"your_username"` below with your GeoNames username.

In [ ]:
from geopy.geocoders import GeoNames

# Create a GeoNames geocoder with your username
geonames = GeoNames(username="your_username")

# Geocode "Vienna" with GeoNames
location = geonames.geocode("Vienna")

print(location.address)
print(location.latitude, location.longitude)

**Question 1**: Compare the results of the three geocoders for "Vienna". Do they return the same address and coordinates?

**Answer**: ...

## Part 2: Place Name Ambiguity

A place name can refer to **multiple positions on the Earth**. So far, each geocoder gave us only one location per place name. But behind the scenes, it found many **candidate locations** and picked one.

### 2.1 Checking All Candidate Locations

To see all candidates, we pass `exactly_one=False` to `geocode()`. It then gives back a **list** of locations instead of one.

We use ArcGIS here, because it returns up to 50 candidates.

In [ ]:
# Get all candidate locations for "Vienna"
candidates = arcgis.geocode("Vienna", exactly_one=False)

print("Number of candidates:", len(candidates))

# Print each candidate
for location in candidates:
    print(location.address, location.latitude, location.longitude)

A list of coordinates is hard to read. Let's plot the candidates on a map with [folium](https://python-visualization.github.io/folium/), a package for interactive web maps.

To do this, we:
- `folium.Map()`: create a map, centered at `location=[latitude, longitude]` and zoomed out with `zoom_start=2` to show the whole world
- `folium.Marker()`: create a marker at a candidate's coordinates, with its address as a `popup` (click a marker to see it)
- `.add_to(m)`: add the marker to our map `m`

To show a map, we pass it to `display()` from `IPython.display`.

In [ ]:
import folium
from IPython.display import display

# Create a world map
m = folium.Map(location=[20, 0], zoom_start=2)

# Add one marker for each candidate
for location in candidates:
    folium.Marker([location.latitude, location.longitude], popup=location.address).add_to(m)

# Show the map
display(m)

We'll repeat these steps for other place names. So, as in Lab 3, we put them into a **function**: it takes a place name, prints the number of candidates, and gives back the map.

In [ ]:
def map_candidates(place_name):
    """
    Plot all candidate locations of a place name on a world map
    """
    candidates = arcgis.geocode(place_name, exactly_one=False)
    print("Number of candidates:", len(candidates))

    m = folium.Map(location=[20, 0], zoom_start=2)
    for location in candidates:
        folium.Marker([location.latitude, location.longitude], popup=location.address).add_to(m)
    return m

# Try the function on "Vienna"
display(map_candidates("Vienna"))

**Task**: Now it's your turn. Use `display(map_candidates())` to find a place name that is **as little ambiguous as possible**, i.e., with as few candidates as possible.

Hint: Try more specific place names, e.g., road names (like "Kärntner Straße"), buildings, or landmarks.

In [ ]:
# Your code here

**Question 2**: Which place name did you find, and how many candidates does it have?

**Answer**: ...

### 2.2 Adding Contextual Clues

Contextual clues help a geocoder decide which place is meant, and many of them are **geospatial**: a country, a state, a county, … (e.g., World > European Union > Austria > Vienna).

We can give such clues to the geocoder by simply adding them to the place name, separated by commas. Now we go back to `exactly_one=True`, so the geocoder gives back only its **best** candidate. Let's see whether the clues change it.

In [ ]:
# No clue
location = arcgis.geocode("Vienna", exactly_one=True)
print(location.address, location.latitude, location.longitude)

In [ ]:
# Clue: country
location = arcgis.geocode("Vienna, Austria", exactly_one=True)
print(location.address, location.latitude, location.longitude)

In [ ]:
# Clue: state
location = arcgis.geocode("Vienna, Virginia", exactly_one=True)
print(location.address, location.latitude, location.longitude)

**Task**: Now it's your turn. Repeat the three steps above for "Vienna" with **Nominatim** and **GeoNames**.

In [ ]:
# Your code here

**Question 3**: How do the geospatial clues change the result of each geocoder? Do the three geocoders react to the clues in the same way?

**Answer**: ...

## Part 3: Evaluating Geocoding

Any tool that appears to perform well still needs to be carefully evaluated. For geocoding, we compare the coordinates returned by a geocoder with the **ground-truth** coordinates.

Let's pretend we already have two lists for five place names:
- `ground_truth`: the correct coordinates, annotated by humans
- `geocoded`: the coordinates returned by a geocoder

Each point is a `(latitude, longitude)` pair, and the two lists are in the same order.

In [ ]:
# Ground-truth coordinates
ground_truth = [
    (48.2083, 16.3725),   # Vienna
    (47.8095, 13.0550),   # Salzburg
    (47.0707, 15.4395),   # Graz
    (47.2692, 11.4041),   # Innsbruck
    (33.6609, -95.5555),  # Paris, Texas
]

# Geocoded coordinates
geocoded = [
    (48.2092, 16.3728),   # Vienna
    (47.8005, 13.0444),   # Salzburg
    (47.0719, 15.4383),   # Graz
    (47.2654, 11.3928),   # Innsbruck
    (48.8566, 2.3522),    # Paris, France
]

### 3.1 Computing Error Distances

The **error distance** is the distance between the ground-truth location and the geocoded output.

Since our coordinates are latitudes and longitudes (a geographic coordinate system), we use the **great-circle distance**, i.e., the shortest distance between two points on the surface of a sphere. In GeoPy, `great_circle(point1, point2).km` gives this distance in kilometers.

In [ ]:
from geopy.distance import great_circle

# Error distance for the first place name (Vienna)
print(great_circle(ground_truth[0], geocoded[0]).km)

Now we compute the error distance for all place names. `zip(ground_truth, geocoded)` pairs up the two lists, so in each round of the loop we get one ground-truth point `truth` and its geocoded point `output`.

In [ ]:
# Compute the error distance for each place name
error_distances = []
for truth, output in zip(ground_truth, geocoded):
    error_distances.append(great_circle(truth, output).km)

print(error_distances)

### 3.2 Computing MED and MdnED

Now we summarize all error distances with the two metrics from the lecture:
- **MED** (Mean Error Distance): the mean of the error distances
- **MdnED** (Median Error Distance): the median of the error distances

We use the [NumPy](https://numpy.org/) package, which has a `mean()` and a `median()` function. By convention, we import it under the short name `np`.

In [ ]:
import numpy as np

# Compute MED and MdnED
med = np.mean(error_distances)
mdned = np.median(error_distances)

print("MED:  ", med, "km")
print("MdnED:", mdned, "km")

**Question 4**: Why are MED and MdnED so different here? Which place name causes this?

**Answer**: ...